Stock Prediction

In [339]:
#basic pakages needed

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as scp 
import seaborn as sns

# Packages needed for the preprocessing of the data 

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import  OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# Packages needed for maintaining

from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

# Packages for the Model Algorithms

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNet


# Packages for Ensemble learning

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import StackingRegressor

# Packages for Metrics 

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold


In [340]:
import sys
print(sys.executable)

c:\Users\HP\anaconda3\python.exe


In [341]:
!{sys.executable} -m pip install yfinance

In [342]:
import yfinance as yf
from datetime import datetime, timedelta

data = yf.download(
    "SOUTHBANK.NS",
    start="2010-01-01",
    end=(datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d"),
    interval="1d"
)

data.to_csv("infosys_data_@.csv")

[*********************100%***********************]  1 of 1 completed


In [343]:
rawdata=pd.read_csv("infosys_data.csv")

In [344]:
rawdata["Tomorrow"]=rawdata["Close"].shift(-1)

In [345]:
rawdata["Target"]=(rawdata["Tomorrow"] > rawdata["Close"]).astype("int32")

In [346]:
rawdata.head()

,Price,Close,High,Low,Open,Volume,Tomorrow,Target
0,2010-01-04,217.472534,218.920900,216.032480,217.256102,4069264,218.200928,1
1,2010-01-05,218.200928,218.979209,216.432080,218.088546,6895528,215.016968,0
2,2010-01-06,215.016968,223.332625,214.351049,223.332625,6817288,210.184937,0
3,2010-01-07,210.184937,215.308357,209.614735,215.308357,10892600,205.119751,0
4,2010-01-08,205.119751,210.039237,204.603668,209.681301,12649312,207.238235,1


In [347]:
rawdata=rawdata.drop(columns="Price")

Build ML 

In [348]:
from sklearn.ensemble import RandomForestClassifier

RF=RandomForestClassifier(min_samples_split=100,n_estimators=100,random_state=1)


In [349]:
predic=["Close","High","Low","Open","Volume"]


In [350]:
def predict(train_data,test_data,predic,model):
    rf_model=model.fit(train_data[predic],train_data["Target"])
    y_pred=rf_model.predict(test_data[predic])
    y_prediction=pd.Series(y_pred,index=test_data.index,name="Prediction")
    combine_y_data=pd.concat([test_data["Target"],y_prediction],axis=1)
    return combine_y_data

In [351]:
def backtest(data,model,predic,start_value=2500,step=250):
    all_prediction=[]

    for i in range(start_value,data.shape[0],step):
        train_data_2=rawdata.iloc[0:i].copy()
        test_data_2=rawdata.iloc[i:(i+step)].copy()
        # simply enna na from the 0 la irundu ella datavum train ku podurom 0 to 2500 before 
        # then 2500 ku munadi iruka ellame trainlaa irukum 
        # and train la iruka data ku next year data matum test lairukum 
        # instead of taking 100 test it is very useful for taking only the next years prediction 
        prediction=predict(train_data_2,test_data_2,predic,model)
        all_prediction.append(prediction)
    return pd.concat(all_prediction)

In [352]:
main_prediction=backtest(rawdata,RF,predic)

In [353]:
from sklearn.metrics import precision_score
precision=precision_score(main_prediction["Target"],main_prediction["Prediction"])

In [354]:
precision

0.5097765363128491

In [355]:
main_prediction["Prediction"].value_counts()

Prediction
0    895
1    716
Name: count, dtype: int64

In [356]:
print(predic)

['Close', 'High', 'Low', 'Open', 'Volume']


In [357]:
pred_data=[46.32,46.45,45.40,45.68,8354307]
pred_real_wrld=RF.predict([pred_data])
pred_real_wrld

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


array([1], dtype=int32)